In [1]:
# ==============================
# CELL 1: Install libraries
# ==============================
!pip install neurokit2 mne pymatreader openneuro-py -q
print("Libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.9/213.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.9/809.9 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.2 MB/s eta 0:00:00
Libraries installed


In [2]:
# ==============================
# CELL 2: Config — EDIT THESE PATHS to match your Kaggle input datasets
# ==============================
import os

target_dir = "/kaggle/working/ds003838"
CHECKPOINT_FILE = "/kaggle/working/external_validation_results_v4.csv"

# Update these to your actual Kaggle input dataset paths
EEG_MODEL_PATH = "/kaggle/input/datasets/himanshubendale000/neuropredict-model7/final_eeg_model.pt (1)"
ECG_MODEL_PATH = "/kaggle/input/datasets/himanshubendale000/neuropredict-model7/final_ecg_model (1).pkl"
# v4 NEW: this file is produced by the training notebook (v4) — upload it alongside the
# two model files above. If it's missing, this notebook still runs, but only with a
# printed warning instead of a verified match (see CELL 4 below).
PREPROCESSING_CONFIG_PATH = "/kaggle/input/datasets/himanshubendale000/neuropredict-model7/preprocessing_config.json"

N_SUBJECTS = 65          # how many subjects to run this pass
MIN_REST_SEC = 60        # skip subject if usable rest duration is below this (too short for stable HRV)
MIN_MEMORY_SEC = 60      # same, for pooled memory-trial duration

# ---- v3: window length must match training (main notebook: window_sec=4, overlap_sec=2) ----
WINDOW_SEC = 4.0         # was 2.0 in v2 — training model expects 2000 samples @ 500Hz = 4 sec
STEP_SEC = 2.0           # 50% overlap, matches training's overlap_sec=2

# ---- v3: cap pooled memory-condition HRV duration so it's comparable to the rest baseline ----
# (v2 pooled ALL ~9 memory-task blocks = ~68 min total vs a ~3-4 min rest baseline — a ~20x
#  duration mismatch that pushes SDNN/RMSSD far outside the training scaler's range)
TARGET_MEMORY_DURATION_SEC = 200   # roughly matches typical rest-baseline duration seen in this dataset

# ---- v4 NEW: fixed ensemble weights, derived from the CONFIRMED multi-seed training run ----
# (training notebook v4, 5-seed reproducible LOSO on EEGMAT: EEG-only mean=0.6667,
#  ECG-only mean=0.6806 — see training run's Cell 16 output). External subjects don't
# have "folds", so we use one fixed weight = each model's own confirmed mean accuracy,
# normalized. This is computed ENTIRELY from training-side (EEGMAT) numbers — zero
# external (ds003838) data was used to pick these weights, so there is no leakage.
TRAIN_EEG_MEAN_ACC = 0.6667
TRAIN_ECG_MEAN_ACC = 0.6806
W_EEG = TRAIN_EEG_MEAN_ACC / (TRAIN_EEG_MEAN_ACC + TRAIN_ECG_MEAN_ACC)
W_ECG = TRAIN_ECG_MEAN_ACC / (TRAIN_EEG_MEAN_ACC + TRAIN_ECG_MEAN_ACC)

os.makedirs(target_dir, exist_ok=True)
print("Config ready. Target subjects:", N_SUBJECTS)
print(f"EEG window: {WINDOW_SEC}s, step: {STEP_SEC}s | ECG memory duration target: {TARGET_MEMORY_DURATION_SEC}s")
print(f"Fixed ensemble weights -> W_EEG={W_EEG:.4f}, W_ECG={W_ECG:.4f}")

# ==============================================================================
# v6 NEW: cross-session resume via a Kaggle Dataset (unlike Colab+Drive, /kaggle/working
# is wiped on every new session -- so the checkpoint CSV must be manually round-tripped
# through a Kaggle Dataset to survive across devices/sessions. One-time setup + a 2-step
# habit each session -- see printed instructions below.
# ==============================================================================
# EDIT THIS after you create the dataset (see instructions) -- point it at the checkpoint
# CSV from your MOST RECENT uploaded version, if any:
RESUME_CHECKPOINT_INPUT_PATH = "/kaggle/input/datasets/himanshubendale000/neuropredict-external-checkpoint/external_validation_results_v4.csv"

if os.path.exists(RESUME_CHECKPOINT_INPUT_PATH):
    shutil_copy_needed = not os.path.exists(CHECKPOINT_FILE)
    import shutil as _shutil
    _shutil.copy(RESUME_CHECKPOINT_INPUT_PATH, CHECKPOINT_FILE)
    _resumed_df = pd.read_csv(CHECKPOINT_FILE)
    print(f"\n✅ Resumed from previous session's checkpoint dataset: "
          f"{len(_resumed_df)} subjects already done ({sorted(_resumed_df['subject'].unique())}).")
else:
    print(f"\n⚠️ No previous checkpoint dataset found at {RESUME_CHECKPOINT_INPUT_PATH} -- "
          f"starting fresh this session. This is expected the very first time; see the "
          f"3-step 'save progress across sessions' instructions below.")

print("""
============================================================
HOW TO CARRY PROGRESS ACROSS SESSIONS (do this once, then repeat steps 2-3 each session):

  1. ONE-TIME SETUP: On kaggle.com, go to "Datasets" -> "New Dataset". Create an empty
     private dataset named e.g. "neuropredict-external-checkpoint". Note its URL slug.

  2. END OF EVERY SESSION (before closing / switching device): download
     /kaggle/working/external_validation_results_v4.csv from the notebook's "Output" tab,
     then on kaggle.com go to your "neuropredict-external-checkpoint" dataset -> "New
     Version" -> upload that CSV (overwriting the old one).

  3. START OF EVERY NEW SESSION: click "Add Input" -> search your
     "neuropredict-external-checkpoint" dataset -> add it. Then just re-run this notebook
     top to bottom -- CELL 2 above will auto-detect it, copy it into place, and CELL 9's
     main loop will skip every subject already in it.
============================================================
""")


Config ready. Target subjects: 65
EEG window: 4.0s, step: 2.0s | ECG memory duration target: 200s
Fixed ensemble weights -> W_EEG=0.4948, W_ECG=0.5052

⚠️ No previous checkpoint dataset found at /kaggle/input/datasets/himanshubendale000/neuropredict-external-checkpoint/external_validation_results_v4.csv -- starting fresh this session. This is expected the very first time; see the 3-step 'save progress across sessions' instructions below.

HOW TO CARRY PROGRESS ACROSS SESSIONS (do this once, then repeat steps 2-3 each session):

  1. ONE-TIME SETUP: On kaggle.com, go to "Datasets" -> "New Dataset". Create an empty
     private dataset named e.g. "neuropredict-external-checkpoint". Note its URL slug.

  2. END OF EVERY SESSION (before closing / switching device): download
     /kaggle/working/external_validation_results_v4.csv from the notebook's "Output" tab,
     then on kaggle.com go to your "neuropredict-external-checkpoint" dataset -> "New
     Version" -> upload that CSV (overwriti

In [3]:
# ==============================
# CELL 3: Download participants.tsv + build subject list (EEG and ECG both usable)
# ==============================
import openneuro
import pandas as pd

openneuro.download(
    dataset="ds003838",
    target_dir=target_dir,
    include=["participants.tsv", "dataset_description.json"]
)

participants = pd.read_csv(f"{target_dir}/participants.tsv", sep="\t")
both_available = participants[
    (participants['EEG_excluded'].astype(str).str.strip().str.lower() == 'no') &
    (participants['ECG_excluded'].astype(str).str.strip().str.lower() == 'no')
]

subject_list = both_available['participant_id'].tolist()[:N_SUBJECTS]
print(f"Total subjects with BOTH EEG+ECG available: {len(both_available)}")
print(f"Subjects selected for this run ({len(subject_list)}): {subject_list}")


👋 Hello! This is openneuro-py 2026.7.1. Great to see you! 🤗

   👉 Please report problems 🤯 and bugs 🪲 at
      https://github.com/openneuro-py/openneuro-py/issues

🌍 Preparing to download ds003838 …
📥 Checking 6 files, downloading as needed (5 concurrent downloads). 


Output()

✅ Finished downloading ds003838 (downloaded 6 files and 13.5 kB).
 
🧠 Please enjoy your brains.
 
Total subjects with BOTH EEG+ECG available: 65
Subjects selected for this run (65): ['sub-032', 'sub-033', 'sub-034', 'sub-035', 'sub-036', 'sub-038', 'sub-039', 'sub-040', 'sub-041', 'sub-042', 'sub-043', 'sub-044', 'sub-045', 'sub-046', 'sub-047', 'sub-048', 'sub-049', 'sub-050', 'sub-051', 'sub-052', 'sub-053', 'sub-054', 'sub-055', 'sub-056', 'sub-057', 'sub-058', 'sub-059', 'sub-060', 'sub-061', 'sub-062', 'sub-063', 'sub-064', 'sub-065', 'sub-067', 'sub-068', 'sub-069', 'sub-070', 'sub-071', 'sub-072', 'sub-073', 'sub-074', 'sub-075', 'sub-076', 'sub-077', 'sub-078', 'sub-079', 'sub-080', 'sub-081', 'sub-082', 'sub-083', 'sub-084', 'sub-085', 'sub-086', 'sub-087', 'sub-088', 'sub-089', 'sub-090', 'sub-091', 'sub-092', 'sub-093', 'sub-094', 'sub-095', 'sub-096', 'sub-097', 'sub-098']


In [4]:
# ==============================
# CELL 4: EEGNet architecture + load EEG model + ECG model
# ==============================
import torch
import torch.nn as nn
import pickle
import json as _json

class EEGNet(nn.Module):
    def __init__(self, n_channels=19, n_classes=2):
        super().__init__()
        self.conv1 = nn.Conv1d(n_channels, 32, 7, padding=3); self.bn1 = nn.BatchNorm1d(32); self.pool1 = nn.MaxPool1d(4)
        self.conv2 = nn.Conv1d(32, 64, 5, padding=2); self.bn2 = nn.BatchNorm1d(64); self.pool2 = nn.MaxPool1d(4)
        self.conv3 = nn.Conv1d(64, 128, 3, padding=1); self.bn3 = nn.BatchNorm1d(128); self.pool3 = nn.MaxPool1d(4)
        self.relu = nn.ReLU(); self.dropout = nn.Dropout(0.5)
        self.global_pool = nn.AdaptiveAvgPool1d(1); self.fc = nn.Linear(128, n_classes)
    def forward(self, x):
        x = self.pool1(self.relu(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu(self.bn2(self.conv2(x))))
        x = self.dropout(x)
        x = self.pool3(self.relu(self.bn3(self.conv3(x))))
        x = self.global_pool(x).squeeze(-1)
        return self.fc(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eeg_model = EEGNet(n_channels=19).to(device)
eeg_model.load_state_dict(torch.load(EEG_MODEL_PATH, map_location=device))
eeg_model.eval()
print("EEG model loaded on", device)

with open(ECG_MODEL_PATH, 'rb') as f:
    ecg_bundle = pickle.load(f)
ecg_clf = ecg_bundle['model']
ecg_scaler = ecg_bundle['scaler']
print("ECG model + scaler loaded")

# ---- v4 NEW: verify this notebook's settings actually match what the model was trained
# with, instead of relying on manually keeping two notebooks' hardcoded constants in sync
# (this is exactly how the earlier 4s-vs-2s window bug happened — silently). ----
if os.path.exists(PREPROCESSING_CONFIG_PATH):
    with open(PREPROCESSING_CONFIG_PATH) as f:
        train_config = _json.load(f)
    print("\nLoaded training preprocessing_config.json — checking for a match...")
    checks = [
        ("window_sec", WINDOW_SEC, train_config["window_sec"]),
        ("step_sec", STEP_SEC, train_config["step_sec"]),
        ("n_channels", 19, train_config["n_channels"]),
        ("sfreq_hz", 500, train_config["sfreq_hz"]),
    ]
    mismatches = [c for c in checks if float(c[1]) != float(c[2])]
    for name, here, trained in checks:
        status = "OK" if float(here) == float(trained) else "❌ MISMATCH"
        print(f"  {name}: this notebook={here} | trained with={trained}  [{status}]")
    # v5 NEW: reference is a string, not a number, so it's checked separately from the
    # numeric checks above -- both sides must say "average" or this notebook's EEG input
    # is referenced differently than what the model was trained on.
    this_reference = "average"
    trained_reference = train_config.get("reference", "<not recorded — pre-v5 config>")
    ref_status = "OK" if this_reference == trained_reference else "❌ MISMATCH"
    print(f"  reference: this notebook={this_reference} | trained with={trained_reference}  [{ref_status}]")
    if this_reference != trained_reference:
        mismatches.append(("reference", this_reference, trained_reference))
    if mismatches:
        raise AssertionError(
            f"Preprocessing settings do not match the training config: {mismatches}. "
            f"Fix WINDOW_SEC/STEP_SEC/channel handling above before running any subjects — "
            f"this exact class of silent mismatch already cost real accuracy once."
        )
    print("All checked settings match training config. Proceeding.")
else:
    print(f"\n⚠️ preprocessing_config.json not found at {PREPROCESSING_CONFIG_PATH} — "
          f"upload it alongside the model files (it's exported by the v4 training notebook's "
          f"last cell) so this match can be verified automatically. Proceeding WITHOUT "
          f"verification for now — double-check WINDOW_SEC/STEP_SEC/channel order by hand.")


EEG model loaded on cuda
ECG model + scaler loaded

Loaded training preprocessing_config.json — checking for a match...
  window_sec: this notebook=4.0 | trained with=4.0  [OK]
  step_sec: this notebook=2.0 | trained with=2.0  [OK]
  n_channels: this notebook=19 | trained with=19  [OK]
  sfreq_hz: this notebook=500 | trained with=500  [OK]
  reference: this notebook=average | trained with=average  [OK]
All checked settings match training config. Proceeding.


In [5]:
# ==============================
# CELL 5: Direct S3 download via GraphQL file index (openneuro-py bypass — kept from v1, this part worked well)
# ==============================
import os, time, shutil, gc
import numpy as np
import pandas as pd
import mne
import neurokit2 as nk
import httpx

mne.set_log_level('WARNING')

# Position-matched to training order (T3→T7, T4→T8, T5→P7, T6→P8 — old/new 10-20 naming)
CHANNEL_ORDER_DS003838 = ['Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'T7', 'T8',
                          'C3', 'C4', 'P7', 'P8', 'P3', 'P4', 'O1', 'O2',
                          'Fz', 'Cz', 'Pz']

GRAPHQL_URL = "https://openneuro.org/crn/graphql"
QUERY = '''
query {
  snapshot(datasetId: "ds003838", tag: "1.0.6") {
    id
    files(recursive: true) {
      filename
      urls
      size
    }
  }
}
'''

print("Fetching the dataset's full file index (URLs) (once)...")
resp = httpx.post(GRAPHQL_URL, json={"query": QUERY}, timeout=60)
resp.raise_for_status()
files_data = resp.json()["data"]["snapshot"]["files"]
FILE_INDEX = {f["filename"]: {"url": f["urls"][0], "size": int(f["size"])} for f in files_data}
print(f"Index built for a total of {len(FILE_INDEX)} files")

def download_one_file(relative_path, max_retries=5):
    if relative_path not in FILE_INDEX:
        print(f"  ⚠️ {relative_path} not found in index")
        return False
    info = FILE_INDEX[relative_path]
    url, expected_size = info["url"], info["size"]
    local_path = f"{target_dir}/{relative_path}"
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    if os.path.exists(local_path) and os.path.getsize(local_path) == expected_size:
        return True
    for attempt in range(max_retries):
        try:
            with httpx.stream("GET", url, timeout=120, follow_redirects=True) as r:
                r.raise_for_status()
                with open(local_path, "wb") as f:
                    for chunk in r.iter_bytes(chunk_size=1024 * 1024):
                        f.write(chunk)
            if os.path.getsize(local_path) == expected_size:
                return True
            print(f"  ⚠️ {relative_path}: size mismatch, retry {attempt+1}")
            os.remove(local_path)
        except Exception as e:
            print(f"  ⚠️ {relative_path}: error {e}, retry {attempt+1}")
            if os.path.exists(local_path):
                os.remove(local_path)
        time.sleep(3 * (attempt + 1))
    return False

def download_subject_files(sub):
    needed_files = [
        f"{sub}/eeg/{sub}_task-memory_events.tsv",
        f"{sub}/eeg/{sub}_task-memory_eeg.set",
        f"{sub}/eeg/{sub}_task-rest_events.tsv",
        f"{sub}/eeg/{sub}_task-rest_eeg.set",
        f"{sub}/ecg/{sub}_task-memory_ecg.set",
        f"{sub}/ecg/{sub}_task-rest_ecg.set",
    ]
    all_ok = True
    for rel_path in needed_files:
        print(f"  Downloading: {rel_path} ...")
        ok = download_one_file(rel_path)
        print(f"  {'✅' if ok else '❌'} {rel_path}")
        all_ok = all_ok and ok
    return all_ok

print("Helper functions ready")

Dataset चा संपूर्ण file index (URLs) मिळवतोय (एकदाच)...
एकूण 2288 files चा index तयार झाला
Helper functions ready


In [6]:
# ==============================
# CELL 6: EEG epoching — v3: window length now matches training (4s window, 2s step)
# ==============================
# WINDOW_SEC and STEP_SEC are set in CELL 2 (4.0s / 2.0s, matching the main training
# notebook's window_sec=4, overlap_sec=2 — the model was trained on 2000-sample @ 500Hz
# inputs, and v2's 2-second/1000-sample windows were a silent train/test mismatch.

def load_raw_eeg(sub, task):
    fpath = f"{target_dir}/{sub}/eeg/{sub}_task-{task}_eeg.set"
    raw = mne.io.read_raw_eeglab(fpath, preload=True, verbose=False)
    raw.pick(CHANNEL_ORDER_DS003838)
    raw.reorder_channels(CHANNEL_ORDER_DS003838)
    raw.filter(l_freq=1.0, h_freq=40.0, verbose=False)
    # v5 NEW: re-reference to common average reference, matching the training notebook.
    # WHY: ds003838's native online reference is FCz (confirmed from its Scientific Data
    # paper), while EEGMAT (training data) uses linked-ears. Without harmonizing these,
    # the model sees EEG referenced differently than what it was trained on -- a silent
    # train/test mismatch on top of everything else. Average reference is applied
    # identically here and in the training notebook so both sides land on common ground.
    raw.set_eeg_reference('average', projection=False, verbose=False)
    raw.resample(500, verbose=False)
    return raw

def extract_memory_epochs(sub, raw):
    """One WINDOW_SEC-long window per 'memory *' event onset (steps by STEP_SEC across
    consecutive digit trials, same 50%-overlap style as training). Control-condition
    events are dropped."""
    ev_path = f"{target_dir}/{sub}/eeg/{sub}_task-memory_events.tsv"
    ev = pd.read_csv(ev_path, sep='\t')
    mem_ev = ev[ev['trial_type'].astype(str).str.startswith('memory')]

    data = raw.get_data(); sfreq = raw.info['sfreq']
    win = int(WINDOW_SEC * sfreq)
    epochs = []
    for onset in mem_ev['onset']:
        start = int(onset * sfreq)
        if start + win <= data.shape[1]:
            epochs.append(data[:, start:start + win])
    return np.array(epochs) if epochs else np.empty((0, len(CHANNEL_ORDER_DS003838), win))

def extract_rest_epochs(sub, raw):
    """Sliding WINDOW_SEC windows (stepping by STEP_SEC, 50% overlap) from 0 up to the
    'eyes opened' onset (fallback: whole file if absent) — same sliding-window scheme
    training used on the continuous rest/task recordings."""
    ev_path = f"{target_dir}/{sub}/eeg/{sub}_task-rest_events.tsv"
    data = raw.get_data(); sfreq = raw.info['sfreq']
    end_sample = data.shape[1]
    if os.path.exists(ev_path):
        ev = pd.read_csv(ev_path, sep='\t')
        eo = ev[ev['value'].astype(str) == 'eyes opened']
        if len(eo) > 0:
            end_sample = min(end_sample, int(eo['onset'].iloc[0] * sfreq))
    win = int(WINDOW_SEC * sfreq)
    step = int(STEP_SEC * sfreq)
    epochs = []
    start = 0
    while start + win <= end_sample:
        epochs.append(data[:, start:start + win])
        start += step
    return np.array(epochs) if epochs else np.empty((0, len(CHANNEL_ORDER_DS003838), win))

def normalize_epochs(epochs):
    if len(epochs) == 0:
        return epochs
    mean_ = epochs.mean(axis=2, keepdims=True)
    std_ = epochs.std(axis=2, keepdims=True) + 1e-8
    return ((epochs - mean_) / std_).astype(np.float32)

def predict_eeg(epochs, model, device, batch_size=32):
    if len(epochs) == 0:
        return np.array([])
    probs = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(epochs), batch_size):
            batch = torch.tensor(epochs[i:i+batch_size], dtype=torch.float32).to(device)
            p = torch.softmax(model(batch), dim=1)[:, 1]
            probs.extend(p.cpu().numpy())
    return np.array(probs)

print(f"EEG epoching functions ready (window={WINDOW_SEC}s / step={STEP_SEC}s, matches training)")


EEG epoching functions ready (window=4.0s / step=2.0s, matches training)


In [7]:
# ==============================
# CELL 7: ECG HRV — v3: memory-duration now actually capped to match rest baseline
# ==============================
# v2 grouped contiguous 'memory'-labeled event rows into runs and pooled ALL of them —
# but ds003838's memory task is ~9 nearly-back-to-back blocks with almost no interleaved
# control events, so v2 still ended up pooling the entire ~68-minute session (~4100s) vs
# a ~3-4 min (~200-260s) rest baseline: a ~20x duration mismatch. SDNN/RMSSD scale with
# duration, so this pushed memory-condition features far outside the range the
# ecg_scaler/ecg_clf were fit on (training rest/task SDNN medians were ~60-90ms).
# Fix: stop accumulating runs once pooled duration reaches TARGET_MEMORY_DURATION_SECl
# (set in CELL 2), so both conditions are the same order of magnitude.

def get_rr_from_continuous(ecg_slice, sfreq):
    """Runs peak detection on one CONTINUOUS ECG segment, returns RR-intervals in ms."""
    try:
        signals, info = nk.ecg_process(ecg_slice, sampling_rate=sfreq)
        peaks = info['ECG_R_Peaks']
        if len(peaks) < 3:
            return np.array([])
        return np.diff(peaks) / sfreq * 1000.0
    except Exception as e:
        print(f"    peak detection failed: {e}")
        return np.array([])

def compute_hrv_from_rr(rr_ms):
    """SDNN/RMSSD from a pool of RR-intervals, with a physiological-range filter (300-2000ms)
    to drop misdetected/merged-beat artifacts before computing variability."""
    if len(rr_ms) < 3:
        return np.nan, np.nan, np.nan, 0
    valid = rr_ms[(rr_ms >= 300) & (rr_ms <= 2000)]
    if len(valid) < 3:
        return np.nan, np.nan, np.nan, 0
    mean_hr = 60000.0 / valid.mean()
    sdnn = valid.std(ddof=1)
    rmssd = np.sqrt(np.mean(np.diff(valid) ** 2))
    return mean_hr, sdnn, rmssd, len(valid)

def get_rest_hrv(sub):
    """Continuous eyes-closed segment only (same boundary logic as EEG rest epochs)."""
    fpath = f"{target_dir}/{sub}/ecg/{sub}_task-rest_ecg.set"
    raw_ecg = mne.io.read_raw_eeglab(fpath, preload=True, verbose=False)
    ch = 'ECG' if 'ECG' in raw_ecg.ch_names else raw_ecg.ch_names[0]
    sfreq = raw_ecg.info['sfreq']
    data = raw_ecg.get_data(picks=[ch])[0]

    ev_path = f"{target_dir}/{sub}/eeg/{sub}_task-rest_events.tsv"
    end_sample = len(data)
    if os.path.exists(ev_path):
        ev = pd.read_csv(ev_path, sep='\t')
        eo = ev[ev['value'].astype(str) == 'eyes opened']
        if len(eo) > 0:
            end_sample = min(end_sample, int(eo['onset'].iloc[0] * sfreq))

    segment = data[:end_sample]
    duration_sec = len(segment) / sfreq
    rr = get_rr_from_continuous(segment, sfreq)
    hr, sdnn, rmssd, n_valid = compute_hrv_from_rr(rr)
    del raw_ecg, data, segment; gc.collect()
    return hr, sdnn, rmssd, duration_sec, n_valid

def get_memory_hrv(sub):
    """Pools RR-intervals across CONTIGUOUS memory-trial runs, but STOPS once total pooled
    duration reaches TARGET_MEMORY_DURATION_SEC — so total duration is actually comparable
    to rest, instead of concatenating every run in the ~68-minute session."""
    fpath = f"{target_dir}/{sub}/ecg/{sub}_task-memory_ecg.set"
    raw_ecg = mne.io.read_raw_eeglab(fpath, preload=True, verbose=False)
    ch = 'ECG' if 'ECG' in raw_ecg.ch_names else raw_ecg.ch_names[0]
    sfreq = raw_ecg.info['sfreq']
    data = raw_ecg.get_data(picks=[ch])[0]

    ev_path = f"{target_dir}/{sub}/eeg/{sub}_task-memory_events.tsv"
    ev = pd.read_csv(ev_path, sep='\t').sort_values('onset').reset_index(drop=True)
    ev['is_memory'] = ev['trial_type'].astype(str).str.startswith('memory')

    # group consecutive memory-condition rows into contiguous runs (unchanged from v2)
    runs = []
    run_start = None
    for i, row in ev.iterrows():
        if row['is_memory'] and run_start is None:
            run_start = row['onset']
        elif not row['is_memory'] and run_start is not None:
            run_end = ev.loc[i - 1, 'onset'] + WINDOW_SEC
            runs.append((run_start, run_end))
            run_start = None
    if run_start is not None:
        run_end = ev.iloc[-1]['onset'] + WINDOW_SEC
        runs.append((run_start, run_end))

    all_rr = []
    total_duration = 0.0
    n_runs_used = 0
    for start_t, end_t in runs:
        if total_duration >= TARGET_MEMORY_DURATION_SEC:
            break  # v3: stop once we've matched the rest baseline's order of magnitude
        # if this run would overshoot the target by a lot, truncate it instead of using it whole
        remaining = TARGET_MEMORY_DURATION_SEC - total_duration
        run_end_capped = min(end_t, start_t + remaining) if (end_t - start_t) > remaining else end_t

        start_s, end_s = int(start_t * sfreq), int(run_end_capped * sfreq)
        if end_s <= start_s or end_s > len(data):
            continue
        segment = data[start_s:end_s]
        rr = get_rr_from_continuous(segment, sfreq)
        if len(rr) > 0:
            all_rr.append(rr)
            total_duration += (run_end_capped - start_t)
            n_runs_used += 1

    pooled_rr = np.concatenate(all_rr) if all_rr else np.array([])
    hr, sdnn, rmssd, n_valid = compute_hrv_from_rr(pooled_rr)
    del raw_ecg, data; gc.collect()
    return hr, sdnn, rmssd, total_duration, n_valid, n_runs_used

print(f"ECG HRV functions ready (duration-matched: memory pooling capped at {TARGET_MEMORY_DURATION_SEC}s, segmented, RR-filtered)")


ECG HRV functions ready (duration-matched: memory pooling capped at 200s, segmented, RR-filtered)


In [8]:
# ==============================
# CELL 8: process_one_subject — combines corrected EEG + ECG extraction, PLUS v4 ensemble
# ==============================
from sklearn.metrics import balanced_accuracy_score

def process_one_subject(sub):
    base = f"{target_dir}/{sub}"
    result = {'subject': sub}
    try:
        ok = download_subject_files(sub)
        if not ok:
            print(f"{sub}: download did not complete, skip")
            shutil.rmtree(base, ignore_errors=True)
            return None

        # ---- EEG ----
        raw_mem = load_raw_eeg(sub, 'memory')
        raw_rest = load_raw_eeg(sub, 'rest')
        epochs_memory = normalize_epochs(extract_memory_epochs(sub, raw_mem))
        epochs_rest = normalize_epochs(extract_rest_epochs(sub, raw_rest))
        del raw_mem, raw_rest; gc.collect()

        if len(epochs_memory) == 0 or len(epochs_rest) == 0:
            print(f"{sub}: no usable EEG epochs (memory={len(epochs_memory)}, rest={len(epochs_rest)}), skip")
            shutil.rmtree(base, ignore_errors=True)
            return None

        probs_rest = predict_eeg(epochs_rest, eeg_model, device)
        probs_memory = predict_eeg(epochs_memory, eeg_model, device)
        y_true = np.concatenate([np.zeros(len(probs_rest)), np.ones(len(probs_memory))])
        y_pred = (np.concatenate([probs_rest, probs_memory]) >= 0.5).astype(int)
        balanced_acc = balanced_accuracy_score(y_true, y_pred)

        result['eeg_n_rest_epochs'] = len(epochs_rest)
        result['eeg_n_memory_epochs'] = len(epochs_memory)
        result['eeg_mean_p_rest'] = probs_rest.mean()
        result['eeg_mean_p_memory'] = probs_memory.mean()
        result['eeg_acc_rest'] = (probs_rest < 0.5).mean()
        result['eeg_acc_memory'] = (probs_memory >= 0.5).mean()
        result['eeg_balanced_acc'] = balanced_acc          # epoch-level (kept for comparison to v3)

        # v4 NEW: condition-level EEG correctness (2 points/subject: rest, memory),
        # same granularity ECG uses below — this is what actually gets compared to
        # ECG-only and Ensemble in the final summary, so it's an apples-to-apples comparison.
        result['eeg_rest_correct'] = int(result['eeg_mean_p_rest'] < 0.5)
        result['eeg_memory_correct'] = int(result['eeg_mean_p_memory'] >= 0.5)
        result['eeg_condition_balanced_acc'] = (result['eeg_rest_correct'] + result['eeg_memory_correct']) / 2
        del epochs_rest, epochs_memory; gc.collect()

        # ---- ECG ----
        hr_rest, sdnn_rest, rmssd_rest, dur_rest, n_rest = get_rest_hrv(sub)
        hr_mem, sdnn_mem, rmssd_mem, dur_mem, n_mem, n_runs = get_memory_hrv(sub)

        result['ecg_hr_rest'] = hr_rest
        result['ecg_hr_memory'] = hr_mem
        result['ecg_sdnn_rest'] = sdnn_rest
        result['ecg_sdnn_memory'] = sdnn_mem
        result['ecg_rmssd_rest'] = rmssd_rest
        result['ecg_rmssd_memory'] = rmssd_mem
        result['rest_duration_sec'] = dur_rest
        result['memory_duration_sec'] = dur_mem
        result['memory_n_runs'] = n_runs
        result['rest_n_valid_rr'] = n_rest
        result['memory_n_valid_rr'] = n_mem
        result['hrv_duration_ok'] = bool(dur_rest >= MIN_REST_SEC and dur_mem >= MIN_MEMORY_SEC)

        ecg_available = result['hrv_duration_ok'] and not (np.isnan(hr_rest) or np.isnan(hr_mem))
        if ecg_available:
            X_test = pd.DataFrame({
                'mean_HR_bpm': [hr_rest, hr_mem],
                'SDNN_ms': [sdnn_rest, sdnn_mem],
                'RMSSD_ms': [rmssd_rest, rmssd_mem]
            })
            X_test_scaled = ecg_scaler.transform(X_test)
            ecg_probs = ecg_clf.predict_proba(X_test_scaled)[:, 1]
            result['ecg_p_rest'] = ecg_probs[0]
            result['ecg_p_memory'] = ecg_probs[1]
            result['ecg_rest_correct'] = int(ecg_probs[0] < 0.5)
            result['ecg_memory_correct'] = int(ecg_probs[1] >= 0.5)
        else:
            print(f"  ⚠️ {sub}: HRV duration too short or invalid (rest={dur_rest:.0f}s, memory={dur_mem:.0f}s) — ECG prediction skipped, not silently included")
            result['ecg_p_rest'] = np.nan
            result['ecg_p_memory'] = np.nan
            result['ecg_rest_correct'] = np.nan
            result['ecg_memory_correct'] = np.nan

        # ---- v4 NEW: Ensemble ----
        # Uses the fixed W_EEG/W_ECG weights from CELL 2 (derived purely from training-side
        # confirmed accuracy — no external data involved in choosing them). Falls back to
        # EEG-only if this subject's ECG wasn't usable, and flags that fallback explicitly
        # rather than silently treating it as a "real" ensemble result.
        if ecg_available:
            ens_p_rest = W_EEG * result['eeg_mean_p_rest'] + W_ECG * result['ecg_p_rest']
            ens_p_memory = W_EEG * result['eeg_mean_p_memory'] + W_ECG * result['ecg_p_memory']
            result['ensemble_used_ecg'] = True
        else:
            ens_p_rest = result['eeg_mean_p_rest']
            ens_p_memory = result['eeg_mean_p_memory']
            result['ensemble_used_ecg'] = False

        result['ensemble_p_rest'] = ens_p_rest
        result['ensemble_p_memory'] = ens_p_memory
        result['ensemble_rest_correct'] = int(ens_p_rest < 0.5)
        result['ensemble_memory_correct'] = int(ens_p_memory >= 0.5)
        result['ensemble_balanced_acc'] = (result['ensemble_rest_correct'] + result['ensemble_memory_correct']) / 2

        shutil.rmtree(base, ignore_errors=True)
        print(f"{sub}: DONE - EEG(cond)={result['eeg_condition_balanced_acc']:.2f} | "
              f"Ensemble={result['ensemble_balanced_acc']:.2f} (used_ecg={result['ensemble_used_ecg']}) | "
              f"ECG dur rest={dur_rest:.0f}s mem={dur_mem:.0f}s ({n_runs} runs) "
              f"rest_ok={result['ecg_rest_correct']} mem_ok={result['ecg_memory_correct']}")
        return result
    except Exception as e:
        print(f"{sub}: ERROR - {e}")
        shutil.rmtree(base, ignore_errors=True)
        return None

print("process_one_subject ready (v4: now also computes ensemble_balanced_acc)")


process_one_subject ready (v4: now also computes ensemble_balanced_acc)


In [9]:
# ==============================
# CELL 9: Main loop - checkpointed, resume-safe
# (re-run this cell any time — already-done subjects are skipped automatically)
# ==============================
if os.path.exists(CHECKPOINT_FILE):
    existing_df = pd.read_csv(CHECKPOINT_FILE)
    already_done = set(existing_df['subject'].unique())
    print(f"Aadhich kelele subjects ({len(already_done)}): {sorted(already_done)}")
else:
    already_done = set()
    print("Navin start - koni subject aadhi zala nahiye")

remaining_subjects = [s for s in subject_list if s not in already_done]
print(f"\nBaki rahilele subjects ({len(remaining_subjects)}): {remaining_subjects}")

for sub in remaining_subjects:
    print(f"\n{'='*50}")
    print(f"Processing {sub}...")
    print('='*50)
    res = process_one_subject(sub)
    if res is not None:
        res_df = pd.DataFrame([res])
        header_needed = not os.path.exists(CHECKPOINT_FILE)
        res_df.to_csv(CHECKPOINT_FILE, mode='a', header=header_needed, index=False)
        print(f"{sub} checkpoint madhe save zala")

print("\n\nALL DONE (or resumed run complete)")

Navin start - koni subject aadhi zala nahiye

Baki rahilele subjects (65): ['sub-032', 'sub-033', 'sub-034', 'sub-035', 'sub-036', 'sub-038', 'sub-039', 'sub-040', 'sub-041', 'sub-042', 'sub-043', 'sub-044', 'sub-045', 'sub-046', 'sub-047', 'sub-048', 'sub-049', 'sub-050', 'sub-051', 'sub-052', 'sub-053', 'sub-054', 'sub-055', 'sub-056', 'sub-057', 'sub-058', 'sub-059', 'sub-060', 'sub-061', 'sub-062', 'sub-063', 'sub-064', 'sub-065', 'sub-067', 'sub-068', 'sub-069', 'sub-070', 'sub-071', 'sub-072', 'sub-073', 'sub-074', 'sub-075', 'sub-076', 'sub-077', 'sub-078', 'sub-079', 'sub-080', 'sub-081', 'sub-082', 'sub-083', 'sub-084', 'sub-085', 'sub-086', 'sub-087', 'sub-088', 'sub-089', 'sub-090', 'sub-091', 'sub-092', 'sub-093', 'sub-094', 'sub-095', 'sub-096', 'sub-097', 'sub-098']

Processing sub-032...
  डाउनलोड करतोय: sub-032/eeg/sub-032_task-memory_events.tsv ...
  ✅ sub-032/eeg/sub-032_task-memory_events.tsv
  डाउनलोड करतोय: sub-032/eeg/sub-032_task-memory_eeg.set ...
  ✅ sub-032/ee

/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/tmp/ipykernel_22/1419934359.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(fpath, preload=True, verbose=False)
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are

sub-032: DONE - EEG(cond)=1.00 | Ensemble=1.00 (used_ecg=True) | ECG dur rest=227s mem=200s (1 runs) rest_ok=1 mem_ok=1
sub-032 checkpoint madhe save zala

Processing sub-033...
  डाउनलोड करतोय: sub-033/eeg/sub-033_task-memory_events.tsv ...
  ✅ sub-033/eeg/sub-033_task-memory_events.tsv
  डाउनलोड करतोय: sub-033/eeg/sub-033_task-memory_eeg.set ...
  ✅ sub-033/eeg/sub-033_task-memory_eeg.set
  डाउनलोड करतोय: sub-033/eeg/sub-033_task-rest_events.tsv ...
  ✅ sub-033/eeg/sub-033_task-rest_events.tsv
  डाउनलोड करतोय: sub-033/eeg/sub-033_task-rest_eeg.set ...
  ✅ sub-033/eeg/sub-033_task-rest_eeg.set
  डाउनलोड करतोय: sub-033/ecg/sub-033_task-memory_ecg.set ...
  ✅ sub-033/ecg/sub-033_task-memory_ecg.set
  डाउनलोड करतोय: sub-033/ecg/sub-033_task-rest_ecg.set ...
  ✅ sub-033/ecg/sub-033_task-rest_ecg.set


/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/tmp/ipykernel_22/1419934359.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(fpath, preload=True, verbose=False)
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are

sub-033: DONE - EEG(cond)=0.50 | Ensemble=0.50 (used_ecg=True) | ECG dur rest=186s mem=200s (1 runs) rest_ok=1 mem_ok=1
sub-033 checkpoint madhe save zala

Processing sub-034...
  डाउनलोड करतोय: sub-034/eeg/sub-034_task-memory_events.tsv ...
  ✅ sub-034/eeg/sub-034_task-memory_events.tsv
  डाउनलोड करतोय: sub-034/eeg/sub-034_task-memory_eeg.set ...
  ✅ sub-034/eeg/sub-034_task-memory_eeg.set
  डाउनलोड करतोय: sub-034/eeg/sub-034_task-rest_events.tsv ...
  ✅ sub-034/eeg/sub-034_task-rest_events.tsv
  डाउनलोड करतोय: sub-034/eeg/sub-034_task-rest_eeg.set ...
  ✅ sub-034/eeg/sub-034_task-rest_eeg.set
  डाउनलोड करतोय: sub-034/ecg/sub-034_task-memory_ecg.set ...
  ✅ sub-034/ecg/sub-034_task-memory_ecg.set
  डाउनलोड करतोय: sub-034/ecg/sub-034_task-rest_ecg.set ...
  ✅ sub-034/ecg/sub-034_task-rest_ecg.set


...

sub-096: DONE - EEG(cond)=0.50 | Ensemble=0.50 (used_ecg=True) | ECG dur rest=279s mem=200s (1 runs) rest_ok=0 mem_ok=1
sub-096 checkpoint madhe save zala

Processing sub-097...
  डाउनलोड करतोय: sub-097/eeg/sub-097_task-memory_events.tsv ...
  ✅ sub-097/eeg/sub-097_task-memory_events.tsv
  डाउनलोड करतोय: sub-097/eeg/sub-097_task-memory_eeg.set ...
  ✅ sub-097/eeg/sub-097_task-memory_eeg.set
  डाउनलोड करतोय: sub-097/eeg/sub-097_task-rest_events.tsv ...
  ✅ sub-097/eeg/sub-097_task-rest_events.tsv
  डाउनलोड करतोय: sub-097/eeg/sub-097_task-rest_eeg.set ...
  ✅ sub-097/eeg/sub-097_task-rest_eeg.set
  डाउनलोड करतोय: sub-097/ecg/sub-097_task-memory_ecg.set ...
  ✅ sub-097/ecg/sub-097_task-memory_ecg.set
  डाउनलोड करतोय: sub-097/ecg/sub-097_task-rest_ecg.set ...
  ✅ sub-097/ecg/sub-097_task-rest_ecg.set


/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/tmp/ipykernel_22/1419934359.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(fpath, preload=True, verbose=False)
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are

sub-097: DONE - EEG(cond)=0.50 | Ensemble=0.50 (used_ecg=True) | ECG dur rest=251s mem=200s (1 runs) rest_ok=0 mem_ok=1
sub-097 checkpoint madhe save zala

Processing sub-098...
  डाउनलोड करतोय: sub-098/eeg/sub-098_task-memory_events.tsv ...
  ✅ sub-098/eeg/sub-098_task-memory_events.tsv
  डाउनलोड करतोय: sub-098/eeg/sub-098_task-memory_eeg.set ...
  ✅ sub-098/eeg/sub-098_task-memory_eeg.set
  डाउनलोड करतोय: sub-098/eeg/sub-098_task-rest_events.tsv ...
  ✅ sub-098/eeg/sub-098_task-rest_events.tsv
  डाउनलोड करतोय: sub-098/eeg/sub-098_task-rest_eeg.set ...
  ✅ sub-098/eeg/sub-098_task-rest_eeg.set
  डाउनलोड करतोय: sub-098/ecg/sub-098_task-memory_ecg.set ...
  ✅ sub-098/ecg/sub-098_task-memory_ecg.set
  डाउनलोड करतोय: sub-098/ecg/sub-098_task-rest_ecg.set ...
  ✅ sub-098/ecg/sub-098_task-rest_ecg.set


/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/tmp/ipykernel_22/1419934359.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(fpath, preload=True, verbose=False)
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are

sub-098: DONE - EEG(cond)=0.50 | Ensemble=0.50 (used_ecg=True) | ECG dur rest=230s mem=200s (1 runs) rest_ok=1 mem_ok=0
sub-098 checkpoint madhe save zala


ALL DONE (or resumed run complete)


In [10]:
# ==============================
# CELL 10: Final aggregate summary — v4: EEG-only vs ECG-only vs Ensemble (paired, condition-level)
# ==============================
BACKUP_FILE = "/kaggle/working/external_validation_results_v4_BACKUP.csv"

# v4.1: resume-safe against a full session restart (not just the main loop) — if the
# primary checkpoint is gone but a backup exists, use that instead of re-running Cell 9.
if os.path.exists(CHECKPOINT_FILE):
    final_df = pd.read_csv(CHECKPOINT_FILE)
elif os.path.exists(BACKUP_FILE):
    final_df = pd.read_csv(BACKUP_FILE)
    print("⚠️ Main checkpoint missing (likely a session restart) — loaded from backup instead.")
else:
    raise FileNotFoundError(
        "Neither the checkpoint nor the backup CSV exists — you need to run Cell 9 "
        "(the main 25-subject loop) at least once first."
    )

print(f"Total subjects processed: {final_df['subject'].nunique()}")

short_hrv = final_df[~final_df['hrv_duration_ok']]
if len(short_hrv) > 0:
    print(f"\n⚠️ {len(short_hrv)} subjects had too-short HRV duration — ECG/ensemble for them fell back to EEG-only:")
    print(short_hrv[['subject', 'rest_duration_sec', 'memory_duration_sec']])

print("\n" + "="*70)
print("PER-SUBJECT RESULTS (condition-level: 2 points/subject, rest+memory)")
print("="*70)
print(final_df[['subject', 'eeg_condition_balanced_acc', 'ensemble_balanced_acc', 'ensemble_used_ecg',
                 'ecg_rest_correct', 'ecg_memory_correct']].to_string(index=False))

paired = final_df[final_df['ensemble_used_ecg'] == True].copy()
n_fallback = (final_df['ensemble_used_ecg'] == False).sum()
if n_fallback > 0:
    print(f"\n⚠️ {n_fallback} subject(s) had no usable ECG — excluded from the paired comparison below "
          f"(their ensemble value is just a copy of EEG-only, which isn't a real test of the ensemble).")

print("\n" + "="*70)
print(f"GROUP-LEVEL SUMMARY — paired comparison, n={len(paired)} subjects with real ECG")
print("="*70)
print(f"EEG-only (condition-level) | Mean: {paired['eeg_condition_balanced_acc'].mean()*100:.1f}% "
      f"(+/- {paired['eeg_condition_balanced_acc'].std()*100:.1f}%)")
ecg_bal = (paired['ecg_rest_correct'] + paired['ecg_memory_correct']) / 2
print(f"ECG-only (condition-level) | Mean: {ecg_bal.mean()*100:.1f}% (+/- {ecg_bal.std()*100:.1f}%)")
print(f"Ensemble                   | Mean: {paired['ensemble_balanced_acc'].mean()*100:.1f}% "
      f"(+/- {paired['ensemble_balanced_acc'].std()*100:.1f}%)")

diff = paired['ensemble_balanced_acc'] - paired['eeg_condition_balanced_acc']
n_wins = (diff > 0).sum()
n_ties = (diff == 0).sum()
print(f"\nEnsemble beat EEG-only in {n_wins}/{len(paired)} subjects, tied in {n_ties} "
      f"| mean change: {diff.mean()*100:+.1f} percentage points")

from scipy import stats
if len(paired) >= 5:
    w_stat, w_p = stats.wilcoxon(paired['ensemble_balanced_acc'], paired['eeg_condition_balanced_acc'])
    print(f"Wilcoxon signed-rank (ensemble vs EEG-only, paired, same subjects): p = {w_p:.4f}")
else:
    print("Too few paired subjects for a meaningful Wilcoxon test — report descriptively only.")

n = len(final_df)
successes_eeg = (final_df['eeg_condition_balanced_acc'] >= 0.5).sum()
successes_ens = (final_df['ensemble_balanced_acc'] >= 0.5).sum()
p_eeg = stats.binomtest(successes_eeg, n, p=0.5, alternative='greater').pvalue
p_ens = stats.binomtest(successes_ens, n, p=0.5, alternative='greater').pvalue
print(f"\nEEG-only : subjects with condition_balanced_acc >= 50%: {successes_eeg}/{n}, binomial p={p_eeg:.4f}")
print(f"Ensemble : subjects with balanced_acc >= 50%: {successes_ens}/{n}, binomial p={p_ens:.4f}")
print(f"\n(Ensemble weights used: W_EEG={W_EEG:.3f}, W_ECG={W_ECG:.3f}, fixed from training-side confirmed accuracy)")
print("\nSaved full results at:", CHECKPOINT_FILE)

# v4.1 NEW: save a backup immediately — download this from Kaggle's Output tab right after
# this cell finishes. If the session ever dies, you reload from THIS file, not from the
# 25-subject loop again.
final_df.to_csv(BACKUP_FILE, index=False)
print(f"\n✅ Backup saved to: {BACKUP_FILE} — download it now (Kaggle Output tab) so you never lose this.")

Total subjects processed: 65

PER-SUBJECT RESULTS (condition-level: 2 points/subject, rest+memory)
subject  eeg_condition_balanced_acc  ensemble_balanced_acc  ensemble_used_ecg  ecg_rest_correct  ecg_memory_correct
sub-032                         1.0                    1.0               True                 1                   1
sub-033                         0.5                    0.5               True                 1                   1
sub-034                         1.0                    1.0               True                 1                   1
sub-035                         0.5                    0.5               True                 1                   1
sub-036                         0.5                    0.5               True                 1                   0
sub-038                         0.5                    0.5               True                 1                   0
sub-039                         0.5                    0.5               True            

In [11]:
# ==============================
# CELL 11: Bootstrap 95% Confidence Intervals (subject-level resampling)
# ==============================
import numpy as np

# v4.1: resilience — if this is a fresh session and `paired` isn't in memory, rebuild it
# from the backup CSV instead of forcing a full re-run of Cell 9/10.
if "paired" not in dir():
    BACKUP_FILE = "/kaggle/working/external_validation_results_v4_BACKUP.csv"
    final_df = pd.read_csv(BACKUP_FILE)
    paired = final_df[final_df['ensemble_used_ecg'] == True].copy()
    print("Reloaded `paired` from backup CSV (was missing from memory).")

def bootstrap_ci(values, n_boot=2000, seed=42):
    """values: condition-level accuracy numbers per subject for one model"""
    rng = np.random.RandomState(seed)
    values = np.array(values)
    n = len(values)
    boot_means = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=n, replace=True)
        boot_means.append(sample.mean())
    return np.array(boot_means)

eeg_vals = paired['eeg_condition_balanced_acc'].values
ecg_vals = ((paired['ecg_rest_correct'] + paired['ecg_memory_correct']) / 2).values
ens_vals = paired['ensemble_balanced_acc'].values

boot_eeg = bootstrap_ci(eeg_vals)
boot_ecg = bootstrap_ci(ecg_vals)
boot_ens = bootstrap_ci(ens_vals)

print("="*70)
print(f"BOOTSTRAP 95% CONFIDENCE INTERVALS (n={len(paired)} subjects, 2000 resamples)")
print("="*70)

def report_ci(name, boot_arr, point_est):
    lo, hi = np.percentile(boot_arr, [2.5, 97.5])
    print(f"{name:12s} | Point Estimate: {point_est*100:.1f}% | 95% CI: [{lo*100:.1f}%, {hi*100:.1f}%]")

report_ci("EEG-only", boot_eeg, eeg_vals.mean())
report_ci("ECG-only", boot_ecg, ecg_vals.mean())
report_ci("Ensemble", boot_ens, ens_vals.mean())

diff_boot = boot_ens - boot_eeg
lo_d, hi_d = np.percentile(diff_boot, [2.5, 97.5])
print(f"\nEnsemble - EEG-only difference | 95% CI: [{lo_d*100:+.1f}%, {hi_d*100:+.1f}%]")
if lo_d > 0:
    print("-> Improvement is statistically reliable (CI does not include 0)")
else:
    print("-> Improvement is not statistically reliable (CI includes 0) -- consistent with the Wilcoxon test")

diff_ecg_eeg = boot_ecg - boot_eeg
lo_e, hi_e = np.percentile(diff_ecg_eeg, [2.5, 97.5])
print(f"\nECG-only - EEG-only difference | 95% CI: [{lo_e*100:+.1f}%, {hi_e*100:+.1f}%]")
if lo_e > 0:
    print("-> ECG is genuinely better than EEG (statistically reliable)")
else:
    print("-> ECG's advantage is not yet statistically confirmed")

# v4.1 NEW: the actually-surprising comparison — is ECG-only genuinely better than the Ensemble?
diff_ecg_ens = boot_ecg - boot_ens
lo_x, hi_x = np.percentile(diff_ecg_ens, [2.5, 97.5])
print(f"\nECG-only - Ensemble difference | 95% CI: [{lo_x*100:+.1f}%, {hi_x*100:+.1f}%]")
if lo_x > 0:
    print("-> ECG-only is genuinely statistically better than the Ensemble")
else:
    print("-> ECG's advantage over the Ensemble is not yet confirmed")

BOOTSTRAP 95% CONFIDENCE INTERVALS (n=65 subjects, 2000 resamples)
EEG-only     | Point Estimate: 56.9% | 95% CI: [53.1%, 61.5%]
ECG-only     | Point Estimate: 60.0% | 95% CI: [55.4%, 65.4%]
Ensemble     | Point Estimate: 64.6% | 95% CI: [59.2%, 70.0%]

Ensemble - EEG-only difference | 95% CI: [+3.8%, +12.3%]
-> सुधारणा Statistically Reliable आहे (CI मध्ये 0 समाविष्ट नाही)

ECG-only - EEG-only difference | 95% CI: [-3.1%, +10.0%]
-> ECG चा फायदा Statistically अजून Confirm झालेला नाही

ECG-only - Ensemble difference | 95% CI: [-11.5%, +2.3%]
-> ECG चा Ensemble वरचा फायदा अजून Confirm झालेला नाही


In [12]:
# ==============================
# CELL 12: Continuous-metric analysis (fixes coarse {0,0.5,1} metric problem)
# Uses already-stored continuous probabilities — no reprocessing needed.
# ==============================
from sklearn.metrics import roc_auc_score, confusion_matrix

def build_continuous(df, rest_col, mem_col):
    y_true = np.concatenate([np.zeros(len(df)), np.ones(len(df))])
    y_score = np.concatenate([df[rest_col].values, df[mem_col].values])
    return y_true, y_score

print("="*70)
print("ROC-AUC (continuous, fixes the 0/0.5/1 coarseness)")
print("="*70)
yt, ys_eeg = build_continuous(paired, 'eeg_mean_p_rest', 'eeg_mean_p_memory')
_,  ys_ecg = build_continuous(paired, 'ecg_p_rest', 'ecg_p_memory')
_,  ys_ens = build_continuous(paired, 'ensemble_p_rest', 'ensemble_p_memory')

auc_eeg = roc_auc_score(yt, ys_eeg)
auc_ecg = roc_auc_score(yt, ys_ecg)
auc_ens = roc_auc_score(yt, ys_ens)
print(f"EEG-only AUC: {auc_eeg:.3f} | ECG-only AUC: {auc_ecg:.3f} | Ensemble AUC: {auc_ens:.3f}")

# bootstrap CI on AUC difference (subject-level resample, same style as CELL 11)
def bootstrap_auc_diff(df, col1_rest, col1_mem, col2_rest, col2_mem, n_boot=2000, seed=42):
    rng = np.random.RandomState(seed)
    n = len(df)
    diffs = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        sub = df.iloc[idx]
        yt_b = np.concatenate([np.zeros(n), np.ones(n)])
        try:
            a1 = roc_auc_score(yt_b, np.concatenate([sub[col1_rest].values, sub[col1_mem].values]))
            a2 = roc_auc_score(yt_b, np.concatenate([sub[col2_rest].values, sub[col2_mem].values]))
            diffs.append(a1 - a2)
        except ValueError:
            continue  # skip resamples with only one class
    return np.array(diffs)

diff_auc = bootstrap_auc_diff(paired, 'ensemble_p_rest', 'ensemble_p_memory', 'eeg_mean_p_rest', 'eeg_mean_p_memory')
lo, hi = np.percentile(diff_auc, [2.5, 97.5])
print(f"Ensemble - EEG-only AUC difference | 95% CI: [{lo:+.3f}, {hi:+.3f}]")

print("\n" + "="*70)
print("Effect size (Cohen's d, paired, condition-level balanced accuracy)")
print("="*70)
diff_vals = (paired['ensemble_balanced_acc'] - paired['eeg_condition_balanced_acc']).values
cohens_d = diff_vals.mean() / diff_vals.std(ddof=1)
print(f"Cohen's d (Ensemble vs EEG-only): {cohens_d:.3f}")
print("(0.2=small, 0.5=medium, 0.8=large — Cohen's convention)")

print("\n" + "="*70)
print("Confusion matrices (threshold=0.5, condition-level, n=%d subjects x 2)" % len(paired))
print("="*70)
for name, ys in [("EEG-only", ys_eeg), ("ECG-only", ys_ecg), ("Ensemble", ys_ens)]:
    yp = (ys >= 0.5).astype(int)
    cm = confusion_matrix(yt, yp)
    print(f"\n{name}:\n{cm}  (rows=true[rest,memory], cols=pred[rest,memory])")

ROC-AUC (continuous, fixes the 0/0.5/1 coarseness)
EEG-only AUC: 0.806 | ECG-only AUC: 0.675 | Ensemble AUC: 0.763
Ensemble - EEG-only AUC difference | 95% CI: [-0.095, +0.007]

Effect size (Cohen's d, paired, condition-level balanced accuracy)
Cohen's d (Ensemble vs EEG-only): 0.423
(0.2=small, 0.5=medium, 0.8=large — Cohen's convention)

Confusion matrices (threshold=0.5, condition-level, n=65 subjects x 2)

EEG-only:
[[ 9 56]
 [ 0 65]]  (rows=true[rest,memory], cols=pred[rest,memory])

ECG-only:
[[43 22]
 [30 35]]  (rows=true[rest,memory], cols=pred[rest,memory])

Ensemble:
[[19 46]
 [ 0 65]]  (rows=true[rest,memory], cols=pred[rest,memory])


In [13]:
# ==============================
# CELL 13: Ablation — Fixed 50/50 average vs confidence-weighted vs LOSO-stacked meta-classifier
# Directly addresses the "W_EEG≈W_ECG≈0.5 = basically averaging" concern.
# ==============================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut

# Scheme 1: plain 50/50 average
avg_p_rest = 0.5 * paired['eeg_mean_p_rest'] + 0.5 * paired['ecg_p_rest']
avg_p_memory = 0.5 * paired['eeg_mean_p_memory'] + 0.5 * paired['ecg_p_memory']
avg_acc = (((avg_p_rest < 0.5).astype(int) + (avg_p_memory >= 0.5).astype(int)) / 2)

# Scheme 2: current confidence-weighted (already computed, W_EEG/W_ECG from CELL 2)
weighted_acc = paired['ensemble_balanced_acc']

# Scheme 3: LOSO-stacked meta-classifier (logistic regression on [eeg_prob, ecg_prob],
# trained on all OTHER subjects' 2 points, tested on the held-out subject's 2 points —
# leave-one-subject-out, so no subject's own data leaks into its own prediction)
X_all = np.vstack([
    np.column_stack([paired['eeg_mean_p_rest'], paired['ecg_p_rest']]),
    np.column_stack([paired['eeg_mean_p_memory'], paired['ecg_p_memory']]),
])
y_all = np.concatenate([np.zeros(len(paired)), np.ones(len(paired))])
subj_all = np.concatenate([paired['subject'].values, paired['subject'].values])

stacked_preds = np.zeros(len(y_all))
for subj in paired['subject'].values:
    test_mask = subj_all == subj
    train_mask = ~test_mask
    clf = LogisticRegression(class_weight='balanced', max_iter=1000)
    clf.fit(X_all[train_mask], y_all[train_mask])
    stacked_preds[test_mask] = clf.predict(X_all[test_mask])

n = len(paired)
stacked_rest_correct = (stacked_preds[:n] == 0).astype(int)
stacked_mem_correct = (stacked_preds[n:] == 1).astype(int)
stacked_acc = (stacked_rest_correct + stacked_mem_correct) / 2

print("="*70)
print("ABLATION: three ensemble schemes compared (paired, n=%d subjects)" % n)
print("="*70)
print(f"Fixed 50/50 average      | Mean: {avg_acc.mean()*100:.1f}% (+/- {avg_acc.std()*100:.1f}%)")
print(f"Confidence-weighted (ours)| Mean: {weighted_acc.mean()*100:.1f}% (+/- {weighted_acc.std()*100:.1f}%)")
print(f"LOSO-stacked meta-clf     | Mean: {stacked_acc.mean()*100:.1f}% (+/- {stacked_acc.std()*100:.1f}%)")

max_diff = max(abs(avg_acc.mean() - weighted_acc.mean()),
               abs(avg_acc.mean() - stacked_acc.mean()),
               abs(weighted_acc.mean() - stacked_acc.mean()))
print(f"\nLargest pairwise mean difference across schemes: {max_diff*100:.1f} percentage points")
print("-> If this difference is small, then honestly state that 'weighting scheme sensitivity is low'")
print("   in the paper -- this directly addresses your near-equal-weights concern.")

ABLATION: three ensemble schemes compared (paired, n=65 subjects)
Fixed 50/50 average      | Mean: 63.8% (+/- 22.5%)
Confidence-weighted (ours)| Mean: 64.6% (+/- 22.9%)
LOSO-stacked meta-clf     | Mean: 70.0% (+/- 24.5%)

Largest pairwise mean difference across schemes: 6.2 percentage points
-> जर हा फरक लहान असेल, तर 'weighting scheme sensitivity कमी आहे' असं honestly
   पेपर मध्ये लिही — हेच तुझा near-equal-weights concern थेट address करतं.


In [14]:
# ==============================
# CELL 14: ECG signal-quality investigation — does cleaning up ECG help the ensemble's AUC?
# Three independent fixes tested, using data already in `paired` — no re-download needed.

# ==============================
# CELL 14 (prepend): resume-safe reload — rebuilds everything Cell 14 needs from the
# CSV on disk, WITHOUT re-running Cell 9 (no re-download). Run this first if you get
# "NameError: paired is not defined" or similar after a session restart.
# ==============================
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

CHECKPOINT_FILE = "/kaggle/working/external_validation_results_v4.csv"
BACKUP_FILE = "/kaggle/working/external_validation_results_v4_BACKUP.csv"

if "paired" not in dir():
    if os.path.exists(CHECKPOINT_FILE):
        final_df = pd.read_csv(CHECKPOINT_FILE)
    elif os.path.exists(BACKUP_FILE):
        final_df = pd.read_csv(BACKUP_FILE)
        print("Loaded from BACKUP_FILE (main checkpoint missing).")
    else:
        raise FileNotFoundError(
            "Neither checkpoint nor backup CSV found on disk -- this is the ONE case "
            "where you'd actually need to re-run Cell 9. Check the Kaggle Output tab "
            "for either file before assuming it's gone."
        )
    paired = final_df[final_df['ensemble_used_ecg'] == True].copy()
    print(f"Rebuilt `paired` from disk: {len(paired)} subjects.")

if "W_EEG" not in dir():
    # same fixed constants as CELL 2 -- recomputing is free, no file needed
    TRAIN_EEG_MEAN_ACC = 0.6667
    TRAIN_ECG_MEAN_ACC = 0.6806
    W_EEG = TRAIN_EEG_MEAN_ACC / (TRAIN_EEG_MEAN_ACC + TRAIN_ECG_MEAN_ACC)
    W_ECG = TRAIN_ECG_MEAN_ACC / (TRAIN_EEG_MEAN_ACC + TRAIN_ECG_MEAN_ACC)
    print(f"Rebuilt W_EEG={W_EEG:.4f}, W_ECG={W_ECG:.4f}")

if "yt" not in dir() or "auc_eeg" not in dir():
    def build_continuous(df, rest_col, mem_col):
        y_true = np.concatenate([np.zeros(len(df)), np.ones(len(df))])
        y_score = np.concatenate([df[rest_col].values, df[mem_col].values])
        return y_true, y_score
    yt, ys_eeg = build_continuous(paired, 'eeg_mean_p_rest', 'eeg_mean_p_memory')
    _,  ys_ecg = build_continuous(paired, 'ecg_p_rest', 'ecg_p_memory')
    _,  ys_ens = build_continuous(paired, 'ensemble_p_rest', 'ensemble_p_memory')
    auc_eeg = roc_auc_score(yt, ys_eeg)
    auc_ecg = roc_auc_score(yt, ys_ecg)
    auc_ens = roc_auc_score(yt, ys_ens)
    print(f"Rebuilt AUCs -> EEG: {auc_eeg:.3f} | ECG: {auc_ecg:.3f} | Ensemble: {auc_ens:.3f}")

print("\nAll variables ready — proceed with the rest of CELL 14 below.")# ==============================
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression

# ---- Fix 1: RR-artifact correction (Kubios-style median-filter outlier rejection) ----
# The current 300-2000ms range filter catches gross errors but not local outliers
# (e.g. one missed/extra beat that's still "physiological" in isolation but breaks
# the local RR pattern). This re-derives SDNN/RMSSD from the SAME stored raw RR
# pattern by additionally rejecting points that deviate >20% from a rolling median —
# standard Kubios/HRV-toolkit practice.
def correct_rr_artifacts(rr_ms, threshold_pct=20):
    if len(rr_ms) < 5:
        return rr_ms
    rr = np.array(rr_ms, dtype=float)
    corrected = rr.copy()
    for i in range(1, len(rr) - 1):
        local_median = np.median(rr[max(0, i-2):i+3])
        if abs(rr[i] - local_median) / local_median > (threshold_pct / 100):
            corrected[i] = local_median  # replace outlier with local median
    return corrected

print("="*70)
print("NOTE: Fix 1 requires raw RR arrays, which weren't saved to the checkpoint CSV")
print("(only derived HR/SDNN/RMSSD were kept). To actually test RR-artifact correction,")
print("get_rest_hrv/get_memory_hrv (CELL 7) need to also return the raw `valid` RR array,")
print("and CELL 8 needs to save it -- that requires re-running CELL 9 (the download loop).")
print("Skipping Fix 1 here since it needs a re-run; Fixes 2 and 3 below use only what's")
print("already in `paired`, so they run instantly.")
print("="*70)

# ---- Fix 2: Probability calibration (Platt/sigmoid scaling on ECG's output) ----
# ECG's raw predict_proba may be poorly calibrated -- "confident" probabilities that
# don't actually reflect correctness rate. Recalibrating (via cross-validated sigmoid
# fit) can fix ranking behavior WITHOUT touching the raw signal at all.
from sklearn.linear_model import LogisticRegression as _LR

ecg_probs_raw = np.concatenate([paired['ecg_p_rest'].values, paired['ecg_p_memory'].values])
ecg_labels = np.concatenate([np.zeros(len(paired)), np.ones(len(paired))])

# Fit a simple 1D Platt scaler (logistic regression on the probability itself) via
# leave-one-subject-out to avoid any leakage
platt_calibrated = np.zeros(len(ecg_probs_raw))
n = len(paired)
subj_idx = np.concatenate([np.arange(n), np.arange(n)])  # same subject appears twice (rest,mem)
for i in range(n):
    train_mask = subj_idx != i
    test_mask = subj_idx == i
    platt = _LR()
    platt.fit(ecg_probs_raw[train_mask].reshape(-1, 1), ecg_labels[train_mask])
    platt_calibrated[test_mask] = platt.predict_proba(ecg_probs_raw[test_mask].reshape(-1, 1))[:, 1]

auc_ecg_calibrated = roc_auc_score(ecg_labels, platt_calibrated)
print(f"\nECG-only AUC | raw: {auc_ecg:.3f} -> Platt-calibrated: {auc_ecg_calibrated:.3f}")

# rebuild ensemble using CALIBRATED ecg probs instead of raw, same fixed W_EEG/W_ECG
ens_calibrated = W_EEG * ys_eeg + W_ECG * platt_calibrated
auc_ens_calibrated = roc_auc_score(yt, ens_calibrated)
print(f"Ensemble AUC | original: {auc_ens:.3f} -> with calibrated ECG: {auc_ens_calibrated:.3f}")

# ---- Fix 3: AUC-based re-weighting instead of accuracy-based ----
# W_EEG/W_ECG (CELL 2) were derived from TRAINING-SIDE mean ACCURACY. But accuracy
# and AUC don't have to agree (exactly the mismatch you're seeing). Re-weighting by
# each model's own AUC (still computed only from paired's data here as a quick check --
# for the real paper version this should come from training-side AUC, same
# no-leakage principle as W_EEG/W_ECG) shows whether smarter weighting alone helps.
w_eeg_auc = auc_eeg / (auc_eeg + auc_ecg)
w_ecg_auc = auc_ecg / (auc_eeg + auc_ecg)
ens_auc_weighted = w_eeg_auc * ys_eeg + w_ecg_auc * ys_ecg
auc_ens_auc_weighted = roc_auc_score(yt, ens_auc_weighted)
print(f"\nEnsemble AUC | fixed-accuracy-weighted (original): {auc_ens:.3f}")
print(f"Ensemble AUC | AUC-weighted (W_EEG={w_eeg_auc:.3f}, W_ECG={w_ecg_auc:.3f}): {auc_ens_auc_weighted:.3f}")

print("\n" + "="*70)
print("SUMMARY: which fix (if any) recovers ensemble AUC toward EEG-only's 0.806?")
print("="*70)
print(f"EEG-only alone:                     {auc_eeg:.3f}  <- target to beat/match")
print(f"Ensemble, original (fixed acc-wt):  {auc_ens:.3f}")
print(f"Ensemble, calibrated ECG:           {auc_ens_calibrated:.3f}")
print(f"Ensemble, AUC-weighted:             {auc_ens_auc_weighted:.3f}")


All variables ready — proceed with the rest of CELL 14 below.
NOTE: Fix 1 requires raw RR arrays, which weren't saved to the checkpoint CSV
(only derived HR/SDNN/RMSSD were kept). To actually test RR-artifact correction,
get_rest_hrv/get_memory_hrv (CELL 7) need to also return the raw `valid` RR array,
and CELL 8 needs to save it -- that requires re-running CELL 9 (the download loop).
Skipping Fix 1 here since it needs a re-run; Fixes 2 and 3 below use only what's
already in `paired`, so they run instantly.

ECG-only AUC | raw: 0.675 -> Platt-calibrated: 0.670
Ensemble AUC | original: 0.763 -> with calibrated ECG: 0.766

Ensemble AUC | fixed-accuracy-weighted (original): 0.763
Ensemble AUC | AUC-weighted (W_EEG=0.544, W_ECG=0.456): 0.765

SUMMARY: which fix (if any) recovers ensemble AUC toward EEG-only's 0.806?
EEG-only alone:                     0.806  <- target to beat/match
Ensemble, original (fixed acc-wt):  0.763
Ensemble, calibrated ECG:           0.766
Ensemble, AUC-weighted: 

In [15]:
# ==============================
# CELL 15: Nested strategy selection — fixes the "best-of-5 on the same test set" bias
# Rebuild-safe: only needs `paired`, `W_EEG`, `W_ECG` (rebuild via CELL 14's resume block
# if a session restart wiped memory).
# ==============================
from sklearn.linear_model import LogisticRegression as _LR2

subjects = paired['subject'].values
n_subj = len(subjects)

eeg_rest = paired['eeg_mean_p_rest'].values
eeg_mem  = paired['eeg_mean_p_memory'].values
ecg_rest = paired['ecg_p_rest'].values
ecg_mem  = paired['ecg_p_memory'].values

def fixed_weighted_predict(idx_train, idx_test):
    p_rest = W_EEG * eeg_rest[idx_test] + W_ECG * ecg_rest[idx_test]
    p_mem  = W_EEG * eeg_mem[idx_test]  + W_ECG * ecg_mem[idx_test]
    correct = np.concatenate([(p_rest < 0.5).astype(int), (p_mem >= 0.5).astype(int)])
    return correct

def meta_stacked_predict(idx_train, idx_test):
    X_train = np.vstack([
        np.column_stack([eeg_rest[idx_train], ecg_rest[idx_train]]),
        np.column_stack([eeg_mem[idx_train],  ecg_mem[idx_train]]),
    ])
    y_train = np.concatenate([np.zeros(len(idx_train)), np.ones(len(idx_train))])
    clf = _LR2(class_weight='balanced', max_iter=1000)
    clf.fit(X_train, y_train)

    X_test = np.vstack([
        np.column_stack([eeg_rest[idx_test], ecg_rest[idx_test]]),
        np.column_stack([eeg_mem[idx_test],  ecg_mem[idx_test]]),
    ])
    y_test = np.concatenate([np.zeros(len(idx_test)), np.ones(len(idx_test))])
    preds = clf.predict(X_test)
    correct = (preds == y_test).astype(int)
    return correct

STRATEGIES = {"fixed_weighted": fixed_weighted_predict, "meta_stacked": meta_stacked_predict}

outer_correct = []
chosen_strategy_log = []
all_idx = np.arange(n_subj)

for outer_i in range(n_subj):
    inner_idx = np.delete(all_idx, outer_i)
    outer_idx = np.array([outer_i])

    inner_scores = {name: [] for name in STRATEGIES}
    for inner_test_i in inner_idx:
        inner_train_idx = np.delete(inner_idx, np.where(inner_idx == inner_test_i)[0])
        inner_test_idx = np.array([inner_test_i])
        for name, fn in STRATEGIES.items():
            c = fn(inner_train_idx, inner_test_idx)
            inner_scores[name].append(c.mean())

    inner_means = {name: np.mean(scores) for name, scores in inner_scores.items()}
    winner = max(inner_means, key=inner_means.get)
    chosen_strategy_log.append(winner)

    final_correct = STRATEGIES[winner](inner_idx, outer_idx)
    outer_correct.append(final_correct.mean())

outer_correct = np.array(outer_correct)
chosen_strategy_log = np.array(chosen_strategy_log)

print("="*70)
print("NESTED STRATEGY SELECTION — no test-set reuse, fully leakage-free")
print("="*70)
print(f"Nested-selected accuracy | Mean: {outer_correct.mean()*100:.1f}% "
      f"(+/- {outer_correct.std()*100:.1f}%), n={n_subj} subjects")
print(f"\nHow often each strategy won the inner comparison:")
for name in STRATEGIES:
    n_won = (chosen_strategy_log == name).sum()
    print(f"  {name:16s}: won {n_won}/{n_subj} times ({n_won/n_subj*100:.0f}%)")

print(f"\nCompare against the naive 'best-of-N on the same set' numbers from CELL 13:")
print(f"  Fixed weighted (no selection, CELL 10/13): 64.6%")
print(f"  Meta-stacked (no selection, CELL 13):       70.0%  <- optimistic, test-set reuse")
print(f"  Nested-selected (THIS cell, honest):        {outer_correct.mean()*100:.1f}%")

diff_from_naive_best = outer_correct.mean()*100 - 70.0
print(f"\nGap between naive best-of-N (70.0%) and honest nested-selected "
      f"({outer_correct.mean()*100:.1f}%): {diff_from_naive_best:+.1f} percentage points")
if abs(diff_from_naive_best) > 2:
    print("-> This gap IS the researcher-degrees-of-freedom bias, now actually measured")
    print("   instead of just flagged as a caveat.")
else:
    print("-> Gap is small -- the meta-classifier's advantage looks real even after")
    print("   removing the test-set-reuse bias, not just a lucky best-of-N pick.")

NESTED STRATEGY SELECTION — no test-set reuse, fully leakage-free
Nested-selected accuracy | Mean: 70.0% (+/- 24.5%), n=65 subjects

How often each strategy won the inner comparison:
  fixed_weighted  : won 0/65 times (0%)
  meta_stacked    : won 65/65 times (100%)

Compare against the naive 'best-of-N on the same set' numbers from CELL 13:
  Fixed weighted (no selection, CELL 10/13): 64.6%
  Meta-stacked (no selection, CELL 13):       70.0%  <- optimistic, test-set reuse
  Nested-selected (THIS cell, honest):        70.0%

Gap between naive best-of-N (70.0%) and honest nested-selected (70.0%): +0.0 percentage points
-> Gap is small -- the meta-classifier's advantage looks real even after
   removing the test-set-reuse bias, not just a lucky best-of-N pick.


## Notes (v4)
- **The ensemble is now actually computed and tested here** — this was the single biggest
  gap in v3: the whole point of the project (does EEG+ECG combined beat EEG-only on truly
  external data?) was never actually run. It is now, via `ensemble_balanced_acc`.
- `W_EEG`/`W_ECG` (CELL 2) are fixed, derived only from the training notebook's own
  confirmed 5-seed accuracy numbers — no ds003838 data was used to choose them, so there's
  no leakage in that weight.
- The paired comparison in CELL 10 only uses subjects where real ECG was available
  (`ensemble_used_ecg == True`) — for the rest, "ensemble" is just a copy of EEG-only and
  including them would understate or overstate the ensemble's real effect depending on
  which subjects those happen to be.
- `ecg_rest_correct`/`ecg_memory_correct` are `NaN` (not silently 0 or 1) for any subject
  whose HRV duration fell below `MIN_REST_SEC`/`MIN_MEMORY_SEC` — check `hrv_duration_ok`
  before trusting a subject's ECG numbers.
- `eeg_balanced_acc` (epoch-level, kept from v3) and `eeg_condition_balanced_acc` (v4, new,
  2-points-per-subject) are **not the same number** — the condition-level one is what's
  actually comparable to ECG-only and Ensemble, since all three are assessed the same way
  (rest correct? memory correct? average the two). Don't mix the two when reporting.
- If `PREPROCESSING_CONFIG_PATH` (CELL 2/4) isn't uploaded, the config-consistency check
  in CELL 4 is skipped with a warning rather than blocking the run — but that means
  WINDOW_SEC/STEP_SEC/channel order are unverified. Upload that file if at all possible;
  it's produced automatically by the last cell of the v4 training notebook.
- `memory_n_runs` should mostly be 1-2, and `memory_duration_sec` should land close to
  `TARGET_MEMORY_DURATION_SEC`, same order of magnitude as `rest_duration_sec`. If it's
  still wildly larger (10x+), the run-grouping logic itself needs a second look for that
  subject.
